# 第 8 章 · 记忆（短期：SessionService）

`SessionService` 管**会话**（一次对话的线程 + 它的 state）。书里列了三种后端：

| 实现 | 存哪 | 适合 |
|---|---|---|
| `InMemorySessionService` | 进程内存 | 测试（重启即失） |
| `DatabaseSessionService` | 数据库（如 SQLite） | 要**跨进程持久化** |
| `VertexAiSessionService` | Vertex AI | GCP 生产环境 |

下面不只构造它们，还**验证 SQLite 那份真的持久化**：写完 → 新开一个服务实例 → 还能读到。

kernel 选 `Python (agentic-adk)`。

In [1]:
import sys
from pathlib import Path

# 【适配】定位仓库根；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break

# 通用脚手架都在 common 里：run_once / run_session / run_async / 重试 / 静音
from common import (adk_model, quiet_adk_sync_warnings, retry_call,
                    run_once, run_session, run_async)
quiet_adk_sync_warnings()

try:
    import google.adk  # noqa: F401  —— 只为内核检查
except ModuleNotFoundError:
    raise RuntimeError(
        "本 notebook 需要内核『Python (agentic-adk)』，当前内核里没有 google.adk。\n"
        "VS Code 里点右上角内核名 → 选 Python (agentic-adk) → 重新运行本格。"
    ) from None

import asyncio
import os
import threading

from google.adk.sessions import InMemorySessionService, DatabaseSessionService



# ---- ① 内存版 ----
mem_service = InMemorySessionService()
print("① InMemorySessionService 就绪（进程结束即失）")

# ---- ② SQLite 版：验证持久化 ----
# 【适配】DatabaseSessionService 要求**异步**驱动，所以用 sqlite+aiosqlite（书里写的是 sqlite://）
DB_URL = "sqlite+aiosqlite:///./_session_demo.db"
DB_FILE = "_session_demo.db"
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)                      # 干净开始

svc1 = DatabaseSessionService(db_url=DB_URL)
run_async(svc1.create_session(app_name="demo", user_id="u1", session_id="s1",
                              state={"fav_seat": "window"}))
print("② 已用 svc1 建了 session，写入 state = {'fav_seat': 'window'}")

# 相当于"重启"：换一个新实例读同一个库
svc2 = DatabaseSessionService(db_url=DB_URL)
s = run_async(svc2.get_session(app_name="demo", user_id="u1", session_id="s1"))
print("   新实例 svc2 读回来的 state =", dict(s.state) if s else "(没读到)")

# ---- ③ Vertex AI 版 ----
try:
    from google.adk.sessions import VertexAiSessionService
    print("\n③ VertexAiSessionService 可导入，但需要 GCP 项目 + 认证，本机不实例化。")
except Exception as e:      # noqa: BLE001
    print("\n③ VertexAiSessionService 不可用:", type(e).__name__)

① InMemorySessionService 就绪（进程结束即失）
② 已用 svc1 建了 session，写入 state = {'fav_seat': 'window'}
   新实例 svc2 读回来的 state = {'fav_seat': 'window'}

③ VertexAiSessionService 可导入，但需要 GCP 项目 + 认证，本机不实例化。
